In [2]:
#!pip install rebound

In [4]:
#!pip install reboundx

In [6]:
import rebound
import reboundx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

In [7]:
import math
import scipy.constants as const

In [8]:
sim = rebound.Simulation()
sim.units = ("m", "s", "Msun")

In [9]:
#Math tests
pi = const.pi
G = sim.G
c = const.c
#meter_to_au = 149597870700.0
#c = c / meter_to_au * 60 * 60 * 24 * 365.25
#print(G)
#print(pi)
#print(c)

In [10]:
def seperation_distance_r(m_1, m_2, e_i, f_gw=10.0):
    """
    Wen's formula: separation for a given GW frequency.
    f_gw: gravitational wave frequency in Hz (default 1 Hz)
    """
    f_orb = f_gw / 2.0  # orbital frequency is half GW frequency
    numerator = G * (m_1 + m_2)
    denominator = (2 * pi * f_orb * ((1 - e_i**2)**1.5) / ((1 + e_i)**1.1954))**2
    r = (numerator / denominator)**(1/3)
    return r

In [16]:
def isco_frequency(m_1, m_2):
    """
    Innermost Stable Circular Orbit frequency.
    Both PN and EccentricTD break down beyond this.
    """
    M_total_kg = (m_1 + m_2) * 1.989e30  # solar masses to kg
    G_si = 6.674e-11
    c_si = 3e8
    f_isco = (c_si**3) / (6**(3/2) * np.pi * G_si * M_total_kg)
    return f_isco

In [18]:
def gr_force(sim_pointer):
    sim = sim_pointer.contents
    if sim.N < 2: return

    ps = sim.particles
    p1, p2 = ps[0], ps[1]

    dx = p1.x - p2.x
    dy = p1.y - p2.y
    dz = p1.z - p2.z
    dvx = p1.vx - p2.vx
    dvy = p1.vy - p2.vy
    dvz = p1.vz - p2.vz

    r = np.sqrt(dx**2 + dy**2 + dz**2)
    v_sq = dvx**2 + dvy**2 + dvz**2
    rv_dot = dx*dvx + dy*dvy + dz*dvz

    M = p1.m + p2.m
    GM = G * M

    factor = GM / (c**2 * r**3)
    coeff_r = (4*GM/r - v_sq)
    coeff_v = 4 * rv_dot / r

    ax_rel = factor * (coeff_r * (dx/r) + coeff_v * dvx)
    ay_rel = factor * (coeff_r * (dy/r) + coeff_v * dvy)
    az_rel = factor * (coeff_r * (dz/r) + coeff_v * dvz)

    m1, m2 = p1.m, p2.m
    p1.ax += ax_rel * (m2 / M)
    p1.ay += ay_rel * (m2 / M)
    p1.az += az_rel * (m2 / M)
    p2.ax -= ax_rel * (m1 / M)
    p2.ay -= ay_rel * (m1 / M)
    p2.az -= az_rel * (m1 / M)

In [20]:
def radiation_reaction_force(sim_pointer):
    sim = sim_pointer.contents
    if sim.N < 2: return

    ps = sim.particles
    p1 = ps[0]
    p2 = ps[1]

    # Relative position and velocity
    dx, dy, dz = p1.x - p2.x, p1.y - p2.y, p1.z - p2.z
    dvx, dvy, dvz = p1.vx - p2.vx, p1.vy - p2.vy, p1.vz - p2.vz

    r_sq = dx**2 + dy**2 + dz**2
    r = np.sqrt(r_sq)
    v_sq = dvx**2 + dvy**2 + dvz**2

    # Radial velocity (r_dot)
    rv_dot = dx*dvx + dy*dvy + dz*dvz
    r_dot = rv_dot / r

    # Mass parameters
    m1, m2 = p1.m, p2.m
    M = m1 + m2
    mu = m1 * m2 / M
    eta = mu / M
    
    # Constants (Ensure G and c are accessible here)
    # 2.5PN Acceleration Factor
    A_const = (8.0/5.0) * (G * M * eta) / (r**3) * (G * M / (c**5))

    # Coefficients
    term_A = (17.0/3.0) * (G * M / r) + 3.0 * v_sq
    term_B = v_sq + 3.0 * (G * M / r)

    # Relative Acceleration Vector
    # CORRECTED FORMULA:
    # a = - A_const * ( term_B * v - term_A * r_dot * n )
    
    ax_rel = -A_const * (term_B * dvx - term_A * r_dot * (dx/r))
    ay_rel = -A_const * (term_B * dvy - term_A * r_dot * (dy/r))
    az_rel = -A_const * (term_B * dvz - term_A * r_dot * (dz/r))

    # Apply forces
    p1.ax += ax_rel * (m2 / M)
    p1.ay += ay_rel * (m2 / M)
    p1.az += az_rel * (m2 / M)

    p2.ax -= ax_rel * (m1 / M)
    p2.ay -= ay_rel * (m1 / M)
    p2.az -= az_rel * (m1 / M)

In [22]:
def run_simulation(m_1, m_2, e_i):
    
    sim = rebound.Simulation()
    sim.units = ("m", "s", "Msun")

    r = seperation_distance_r(m_1, m_2, e_i, f_gw=10.0)
    f_isco = isco_frequency(m_1, m_2)

    sim.add(m=m_1)
    sim.add(m=m_2, a=r, e=e_i)
    sim.move_to_com()

    sim.integrator = "ias15"
    def combined_forces(sim_pointer):
        gr_force(sim_pointer)
        radiation_reaction_force(sim_pointer)
    sim.additional_forces = combined_forces
    sim.force_is_velocity_dependent = 1
    
    sim.additional_forces = radiation_reaction_force
    sim.force_is_velocity_dependent = 1

    ps = sim.particles
    ps[0].r = 2 * G * ps[0].m / (c**2)
    ps[1].r = 2 * G * ps[1].m / (c**2)
    sim.collision = "direct"
    sim.collision_resolve = "merge"

    dt_record = 0.1
    waveform_data = []
    time_data = []
    separation_data = []
    eccentricity_data = []
    
    while sim.N == 2:
        sim.integrate(sim.t + dt_record)
    
        if sim.N < 2:
            break
    
        orbit = sim.orbits()[0]
        current_e = orbit.e
        current_r = np.sqrt((ps[1].x-ps[0].x)**2 + (ps[1].y-ps[0].y)**2 + (ps[1].z-ps[0].z)**2)
    
        time_data.append(sim.t)
        separation_data.append(current_r)
        eccentricity_data.append(current_e)
    
        f_orb = orbit.n / (2 * np.pi)
        peak_freq = f_orb * 2 * (1 + orbit.e)**1.1954 / (1 - orbit.e**2)**1.5
    
        if peak_freq > f_isco:
            break
    return np.array(time_data), np.array(separation_data), np.array(eccentricity_data)

In [24]:
def animate_decay(m_1, m_2, e_i):
    time, separation, eccentricity = run_simulation(m_1, m_2, e_i)
    
    # Create figure and axes
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Set axis labels and titles
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Separation (m)")
    ax1.set_title("Separation vs Time")
    
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Eccentricity")
    ax2.set_title("Eccentricity vs Time")
    
    # Set axis limits
    ax1.set_xlim(time.min(), time.max())
    ax1.set_ylim(separation.min(), separation.max() + 0.5)
    
    ax2.set_xlim(time.min(), time.max())
    ax2.set_ylim(eccentricity.min(), eccentricity.max() + 0.05)
    
    # Create empty lines
    line1, = ax1.plot([], [], lw=2, color='blue')
    line2, = ax2.plot([], [], lw=2, color='crimson')
    
    def init():
        line1.set_data([], [])
        line2.set_data([], [])
        return line1, line2
    
    def update(frame):
        # 'frame' will now be the actual index we want to draw up to
        line1.set_data(time[:frame], separation[:frame])
        line2.set_data(time[:frame], eccentricity[:frame])
        return line1, line2
    
    # --- THE FIX: DOWNSAMPLING ---
    total_points = len(time)
    max_frames = 250 # Limit the video to 250 frames
    step_size = max(1, total_points // max_frames)
    frames_to_draw = list(range(0, total_points, step_size))
    if frames_to_draw[-1] != total_points - 1:
       frames_to_draw.append(total_points - 1)    
    #print(f"Downsampling {total_points} points into {len(frames_to_draw)} animation frames...")
    
    # Create animation
    ani = FuncAnimation(
        fig,
        update,
        frames=frames_to_draw, # Use the downsampled list here!
        init_func=init,
        blit=True,
        interval=30
    )
    
    plt.tight_layout()
    plt.close(fig) # Prevent the empty static plot from showing
    
    # --- THE FIX: DISPLAYING IN JUPYTER ---
    print("Rendering animation...")
    html_str = ani.to_jshtml()
    display(HTML(html_str))

In [26]:
import ipywidgets as widgets
from IPython.display import display

print("--- Binary Black Hole Merger Separation and Eccentricity Decay Simulation ---")

# Create visual input boxes
widget_style = {'description_width': 'initial'}
m_1_input = widgets.FloatSlider(value=30.0, min=1, max=100, step=1, description='Mass 1 (Solar mass):', style=widget_style)
m_2_input = widgets.FloatSlider(value=30.0, min=1, max=100, step=1, description='Mass 2 (Solar mass):', style=widget_style)
e_i_input = widgets.FloatSlider(value=0.2, min=0.01, max=0.99, step=0.01, description='Initial Eccentricity:', style=widget_style)
run_button = widgets.Button(description="Run Simulation", button_style='info')
output = widgets.Output()

# Display the widgets on the screen
display(m_1_input, m_2_input, e_i_input, run_button, output)

# RENAME THIS FUNCTION so it doesn't overwrite your physics function!
def on_button_clicked(b):
    with output:
        output.clear_output() # Clears previous runs
        m_1 = m_1_input.value
        m_2 = m_2_input.value
        e_i = e_i_input.value
        
        # Call your animation function (which then safely calls your REAL run_simulation)
        animate_decay(m_1, m_2, e_i)

# Update the click trigger to match the new name
run_button.on_click(on_button_clicked)

--- Binary Black Hole Merger Separation and Eccentricity Decay Simulation ---


FloatSlider(value=30.0, description='Mass 1 (Solar mass):', min=1.0, step=1.0, style=SliderStyle(description_w…

FloatSlider(value=30.0, description='Mass 2 (Solar mass):', min=1.0, step=1.0, style=SliderStyle(description_w…

FloatSlider(value=0.2, description='Initial Eccentricity:', max=0.99, min=0.01, step=0.01, style=SliderStyle(d…

Button(button_style='info', description='Run Simulation', style=ButtonStyle())

Output()